In [2]:
import kagglehub
import pandas as pd
from pathlib import Path

path = Path(kagglehub.dataset_download("samlearner/letterboxd-movie-ratings-data"))

movie_data = pd.read_csv(path / "movie_data.csv", on_bad_lines="skip", engine="python")
ratings_data = pd.read_csv(
    path / "ratings_export.csv", on_bad_lines="skip", engine="python"
)

display(movie_data.head())

,_id,genres,image_url,imdb_id,imdb_link,movie_id,movie_title,original_language,overview,popularity,production_countries,release_date,runtime,spoken_languages,tmdb_id,tmdb_link,vote_average,vote_count,year_released
0,5fc85f606758f69634496fd3,"[""Music"",""Animation""]",film-poster/4/6/4/4/4/0/464440-football-freaks...,NaN,NaN,football-freaks,Football Freaks,en,"Football crazy, football mad. Don’t watch this...",0.600,"[""United Kingdom""]",1971-12-05,0.0,[],535272.0,https://www.themoviedb.org/movie/535272/,0.0,0.0,1971.0
1,5fc85ff26758f696344ace0c,[],film-poster/2/4/5/5/0/0/245500-aftermath-0-230...,tt0586129,http://www.imdb.com/title/tt0586129/maindetails,aftermath-1960,Aftermath,en,Aftermath was the pilot for an unsold TV serie...,0.600,[],1960-04-17,22.0,[],318331.0,https://www.themoviedb.org/movie/318331/,8.0,1.0,1960.0
2,5fc85f606758f69634496fcd,"[""Drama""]",film-poster/9/3/3/1/8/93318-where-chimneys-are...,tt0045731,http://www.imdb.com/title/tt0045731/maindetails,where-chimneys-are-seen,Where Chimneys Are Seen,ja,Gosho’s most celebrated film both in Japan and...,1.568,"[""Japan""]",1953-03-05,108.0,"[""日本語""]",117779.0,https://www.themoviedb.org/movie/117779/,6.6,10.0,1953.0
3,5fc85f606758f69634496fd1,"[""Drama""]",NaN,tt0187327,http://www.imdb.com/title/tt0187327/maindetails,the-musicians-daughter,The Musician's Daughter,en,Carl Wagner's good wife was dying. His heart b...,0.600,"[""United States of America""]",1911-12-12,15.0,[],560377.0,https://www.themoviedb.org/movie/560377/,0.0,0.0,1911.0
4,5fc85f606758f69634496fd4,"[""Documentary""]",film-poster/4/5/4/6/0/3/454603-50-years-of-fab...,tt4769914,http://www.imdb.com/title/tt4769914/maindetails,50-years-of-fabulous,50 Years of Fabulous,en,50 Years of Fabulous recounts the rich history...,0.600,[],2018-05-17,75.0,[],525187.0,https://www.themoviedb.org/movie/525187/,0.0,0.0,2018.0


In [4]:
# Przygotowanie danych (czyszczenie i filtrowanie)
chosen_user = "shiftless"

user_ratings = ratings_data[ratings_data["user_id"] == chosen_user]
movie_data = movie_data.drop_duplicates(subset=["movie_id"])
merged_data = pd.merge(
    user_ratings, movie_data, left_on="movie_id", right_on="movie_id", how="inner"
)
final_dataset = merged_data[["overview", "rating_val"]]
final_dataset = final_dataset.dropna(subset=["overview"])
final_dataset["rating_val"] = final_dataset["rating_val"].astype(float)

print(f"Liczba filmów gotowych do treningu: {len(final_dataset)}")
display(final_dataset.head())

Liczba filmów gotowych do treningu: 3721


,overview,rating_val
0,Sinbad and his crew intercept a homunculus car...,8.0
1,"Will Lockhart arrives in Coronado, an isolated...",8.0
2,A hard-drinking reporter tries to help the emb...,6.0
3,"The French Revolution, 1794. The Marquis de La...",7.0
4,Genial shopkeeper Philip has to endure the con...,7.0


In [5]:
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer

In [7]:
# Przygotowanie danych (tokenizacja)

# konwersja danych na zrozumiałe dla trenera
final_dataset = final_dataset.rename(columns={"rating_val": "label"})
hf_dataset = Dataset.from_pandas(final_dataset)
# podział danych na test i train
hf_dataset = hf_dataset.train_test_split(test_size=0.2, seed=42)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")


def tokenize_function(examples):
    return tokenizer(
        examples["overview"], padding="max_length", truncation=True, max_length=512
    )


tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)

print(tokenized_datasets)

Map: 100%|██████████| 745/745 [00:00<00:00, 5909.08 examples/s]

DatasetDict({
    train: Dataset({
        features: ['overview', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2976
    })
    test: Dataset({
        features: ['overview', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 745
    })
})


In [8]:
# obejście problemu z importowaniem
import torch

if not hasattr(torch, "float8_e8m0fnu"):
    torch.float8_e8m0fnu = torch.float32

In [9]:
import evaluate
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=1
)

metric = evaluate.load("mae")


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.squeeze(predictions)
    return metric.compute(predictions=predictions, references=labels)


# 2 epoki, po każdej epoce model jest zapisywany i próbuje rozwiązać test
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    eval_strategy="epoch",
    learning_rate=2e-5,
    save_strategy="epoch",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    fp16=True,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)


trainer.train()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 12720.03it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Mae
1,3.737206,1.476061,0.931329
2,1.577970,1.380459,0.894536


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.27it/s]


TrainOutput(global_step=1488, training_loss=2.2299887134182836, metrics={'train_runtime': 188.6349, 'train_samples_per_second': 31.553, 'train_steps_per_second': 7.888, 'total_flos': 788431895986176.0, 'train_loss': 2.2299887134182836, 'epoch': 2.0})